In [7]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

#read training data
train = pd.read_csv('/kaggle/input/nexar-collision-prediction/train.csv')
test = pd.read_csv('/kaggle/input/nexar-collision-prediction/test.csv')

In [8]:
safe = train[train['target'] == 0]
unsafe = train[train['target'] == 1]

In [9]:
import cv2

#get frames from videos
def extract_frames(video_path, num_frames):
    #open video
    cap = cv2.VideoCapture(video_path)
    
    #get total num of frames
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    #determine step size
    #divide total num of frames by provided num of frames
    #calculates how many frames to skip between each extracted frame
    step = max(total_frames // num_frames, 1)

    #initialize frames
    frames = []

    #loop through video frames w/ step size
    for i in range(0, total_frames, step):
        #set current frame position
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)

        #read frame
        ret, frame = cap.read()

        #if not read successfully...
        if not ret:
            break
            
        #convert BGR to RGB for display in matplotlib
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        #add frame to frames list
        frames.append(frame)

    #release video once all frames are processed 
    cap.release()
    return frames

In [10]:
def compute_optical_flow(frames):
    flow_features = []
    
    for i in range(len(frames) - 1):
        prev_gray = cv2.cvtColor(frames[i], cv2.COLOR_RGB2GRAY)
        cur_gray = cv2.cvtColor(frames[i+1], cv2.COLOR_RGB2GRAY)
        
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, cur_gray, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )
        
        
        magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        
        # Get overall statistics
        mean_magnitude = np.mean(magnitude)
        max_magnitude = np.max(magnitude)
        std_magnitude = np.std(magnitude)
        
        # Get directional components
        flow_x = flow[..., 0]  
        flow_y = flow[..., 1]  
        
        
        mean_flow_x = np.mean(flow_x)
        mean_flow_y = np.mean(flow_y)
        
    
        flow_features.append([
            mean_magnitude,  
            std_magnitude,   
            max_magnitude,   
            mean_flow_x,     
            mean_flow_y      
        ])
    
   
    if not flow_features:
        return [0, 0, 0, 0, 0, 0]  
    
    flow_array = np.array(flow_features)
    
    
    result = [
        np.mean(flow_array[:, 0]),  
        np.std(flow_array[:, 0]),   
        np.mean(flow_array[:, 1]),  
        np.mean(flow_array[:, 2]),  
        np.mean(flow_array[:, 3]),  
        np.mean(flow_array[:, 4])   
    ]
    
    return result

In [11]:
def extract_edges(video_path, num_frames=10):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total_frames // num_frames, 1)

    for i in range(0, total_frames, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 100, 200)
        frames.append(edges)
    
    cap.release()
    return np.array(frames)

In [12]:
def extract_video_features(video_path, num_frames):
    # Sample exactly num_frames frames
    frames = extract_frames(video_path, num_frames)  

    # Brightness
    brights = [f.mean() for f in frames] if frames else [0]
    bright_mean = np.mean(brights)
    bright_var = np.var(brights)
    
    # Brightness changes (sudden changes could indicate collision)
    bright_changes = [abs(brights[i+1] - brights[i]) for i in range(len(brights)-1)] if len(brights) > 1 else [0]
    max_bright_change = max(bright_changes) if bright_changes else 0
    mean_bright_change = np.mean(bright_changes) if bright_changes else 0

    # Edge features 
    densities = []
    for f in frames:
        gray = cv2.cvtColor(f, cv2.COLOR_RGB2GRAY)
        e = cv2.Canny(gray, 100, 200)
        densities.append((e>0).mean())
    edge_mean = np.mean(densities) if densities else 0
    edge_std = np.std(densities) if densities else 0
    
    # Edge density changes (sudden increases could indicate objects appearing)
    edge_changes = [abs(densities[i+1] - densities[i]) for i in range(len(densities)-1)] if len(densities) > 1 else [0]
    max_edge_change = max(edge_changes) if edge_changes else 0
    
    # Optical flow features
    flow_features = compute_optical_flow(frames)
    
    # Motion change detection (for sudden deceleration/acceleration)
    motion_changes = []
    if len(frames) >= 3:
        for i in range(len(frames)-2):
            # Calculate frame-to-frame difference as a simple motion metric
            diff1 = np.mean(np.abs(frames[i+1].astype(float) - frames[i].astype(float)))
            diff2 = np.mean(np.abs(frames[i+2].astype(float) - frames[i+1].astype(float)))
            # Motion change is the difference between consecutive motion metrics
            change = abs(diff2 - diff1)
            motion_changes.append(change)
    
    max_motion_change = max(motion_changes) if motion_changes else 0
    mean_motion_change = np.mean(motion_changes) if motion_changes else 0
    
    # Return all features
    return flow_features + [
        edge_mean, 
        edge_std, 
        bright_mean, 
        bright_var,
        max_bright_change,
        mean_bright_change,
        max_edge_change,
        max_motion_change,
        mean_motion_change
    ]

In [13]:
#create sample set

import os
import random

#randomly sample 50 safe and 50 unsafe 

safe_sample = safe.sample(n=100, random_state=42)
unsafe_sample = unsafe.sample(n=100, random_state=42)
sample = pd.concat([safe_sample, unsafe_sample]).reset_index(drop=True)

#print("Number of safe sample videos:", len(safe_sample))
#print("Number of unsafe sample videos:", len(unsafe_sample))

In [14]:
feature_dicts = []

video_dir = '/kaggle/input/nexar-collision-prediction/train/'

for _, row in sample.iterrows():
    vid = str(int(row['id'])).zfill(5)
    print(f"Processing #{_+1}/{len(sample)}: video {vid}.mp4") 
    path = os.path.join(video_dir, f"{vid}.mp4")
    

    features = extract_video_features(path, 15)  
    

    feature_dicts.append({
        'video_id': vid,
        'flow_mean_magnitude': features[0],
        'flow_magnitude_std': features[1],
        'flow_std_mean': features[2],
        'flow_max_mean': features[3],
        'flow_x_mean': features[4],
        'flow_y_mean': features[5],
        'edge_mean': features[6],
        'edge_std': features[7],
        'bright_mean': features[8],
        'bright_var': features[9],
        'target': row['target']
    })
    
    print(f"[{_+1}/{len(sample)}] {vid}.mp4 → Features extracted")

features = pd.DataFrame(feature_dicts)

X = features[['flow_mean_magnitude', 'flow_magnitude_std', 'flow_std_mean', 
              'flow_max_mean', 'flow_x_mean', 'flow_y_mean',
              'edge_mean', 'edge_std', 'bright_mean', 'bright_var']]
y = features['target']

Processing #1/200: video 01875.mp4
[1/200] 01875.mp4 → Features extracted
Processing #2/200: video 01071.mp4
[2/200] 01071.mp4 → Features extracted
Processing #3/200: video 01849.mp4
[3/200] 01849.mp4 → Features extracted
Processing #4/200: video 01372.mp4
[4/200] 01372.mp4 → Features extracted
Processing #5/200: video 01853.mp4
[5/200] 01853.mp4 → Features extracted
Processing #6/200: video 01478.mp4
[6/200] 01478.mp4 → Features extracted
Processing #7/200: video 01617.mp4
[7/200] 01617.mp4 → Features extracted
Processing #8/200: video 01696.mp4
[8/200] 01696.mp4 → Features extracted
Processing #9/200: video 01542.mp4
[9/200] 01542.mp4 → Features extracted
Processing #10/200: video 02048.mp4
[10/200] 02048.mp4 → Features extracted
Processing #11/200: video 02121.mp4
[11/200] 02121.mp4 → Features extracted
Processing #12/200: video 01806.mp4
[12/200] 01806.mp4 → Features extracted
Processing #13/200: video 01815.mp4
[13/200] 01815.mp4 → Features extracted
Processing #14/200: video 0199

In [15]:
features = pd.DataFrame(feature_dicts)

X = features[['flow_mean_magnitude', 'flow_magnitude_std', 'flow_std_mean', 
              'flow_max_mean', 'flow_x_mean', 'flow_y_mean',
              'edge_mean', 'edge_std', 'bright_mean', 'bright_var']]
y = features['target']

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [17]:
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Define different models to try
models = {
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42),
    'NeuralNet': MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42)
}

# Evaluate each model
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:,1]
    
    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    
    results[name] = {'Accuracy': acc, 'AUC': auc}
    print(f"{name}: Accuracy = {acc:.4f}, AUC = {auc:.4f}")

RandomForest: Accuracy = 0.4500, AUC = 0.5088
GradientBoosting: Accuracy = 0.5000, AUC = 0.5400
AdaBoost: Accuracy = 0.4500, AUC = 0.5350
SVC: Accuracy = 0.6500, AUC = 0.4062
XGBoost: Accuracy = 0.5250, AUC = 0.5650
[LightGBM] [Info] Number of positive: 80, number of negative: 80
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001910 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 550
[LightGBM] [Info] Number of data points in the train set: 160, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [18]:
# Calibrate SVC for better probability estimates

from sklearn.calibration import CalibratedClassifierCV
base_svc = SVC(kernel='rbf', random_state=42)
calibrated_svc = CalibratedClassifierCV(base_svc, cv=5, method='sigmoid')
calibrated_svc.fit(X_train_scaled, y_train)

# Evaluate
y_pred = calibrated_svc.predict(X_val_scaled)
y_prob = calibrated_svc.predict_proba(X_val_scaled)[:,1]

print(f"Calibrated SVC: Accuracy = {accuracy_score(y_val, y_pred):.4f}, AUC = {roc_auc_score(y_val, y_prob):.4f}")

Calibrated SVC: Accuracy = 0.5000, AUC = 0.4800


In [19]:
# Tune SVC hyperparameters
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'class_weight': [None, 'balanced']
}

# Grid search (this will take some time to run)
grid = GridSearchCV(SVC(probability=True, random_state=42), 
                   param_grid, 
                   cv=5, 
                   scoring='accuracy')
grid.fit(X_train_scaled, y_train)

# Best parameters
print("Best parameters:", grid.best_params_)
best_svc = grid.best_estimator_

# Evaluate best model
best_pred = best_svc.predict(X_val_scaled)
best_prob = best_svc.predict_proba(X_val_scaled)[:,1]

print(f"Tuned SVC: Accuracy = {accuracy_score(y_val, best_pred):.4f}, AUC = {roc_auc_score(y_val, best_prob):.4f}")

from sklearn.ensemble import VotingClassifier

# Create the original SVC model with the same parameters
original_svc = SVC(probability=True, random_state=42)

# Train it on scaled features
original_svc.fit(X_train_scaled, y_train)

# Create XGBoost model
xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# Create weighted ensemble with original SVC and XGBoost
combined_ensemble = VotingClassifier(
    estimators=[
        ('svc', original_svc),  # Use the original SVC model
        ('xgb', xgb_model)
    ],
    voting='soft',  # Use probability estimates
    weights=[0.7, 0.3]  # Give more weight to SVC since it has better accuracy
)

# Train ensemble
combined_ensemble.fit(X_train_scaled, y_train)

# Evaluate
combined_pred = combined_ensemble.predict(X_val_scaled)
combined_prob = combined_ensemble.predict_proba(X_val_scaled)[:,1]

print(f"Combined Ensemble: Accuracy = {accuracy_score(y_val, combined_pred):.4f}, AUC = {roc_auc_score(y_val, combined_prob):.4f}")

Best parameters: {'C': 1, 'class_weight': None, 'gamma': 'scale', 'kernel': 'rbf'}
Tuned SVC: Accuracy = 0.5250, AUC = 0.4750
Combined Ensemble: Accuracy = 0.5750, AUC = 0.5200


In [20]:
from sklearn.metrics      import accuracy_score, roc_auc_score

#evaluate
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:,1]

print("Val accuracy:", accuracy_score(y_val, y_pred))
print("Val ROC AUC  :", roc_auc_score(y_val, y_prob))

Val accuracy: 0.525
Val ROC AUC  : 0.5625


In [21]:
from sklearn.ensemble import VotingClassifier

# Create the original SVC model 
original_svc = SVC(probability=True, random_state=42)

original_svc.fit(X_train_scaled, y_train)

# Create XGBoost model
xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# Create weighted ensemble with original SVC and XGBoost
combined_ensemble = VotingClassifier(
    estimators=[
        ('svc', original_svc),  
        ('xgb', xgb_model)
    ],
    voting='soft', 
    weights=[0.7, 0.3]  # Give more weight to SVC since it has better accuracy
)

# Train ensemble
combined_ensemble.fit(X_train_scaled, y_train)

# Evaluate
combined_pred = combined_ensemble.predict(X_val_scaled)
combined_prob = combined_ensemble.predict_proba(X_val_scaled)[:,1]

print(f"Combined Ensemble: Accuracy = {accuracy_score(y_val, combined_pred):.4f}, AUC = {roc_auc_score(y_val, combined_prob):.4f}")

Combined Ensemble: Accuracy = 0.5750, AUC = 0.5200


In [22]:
print(features[['flow_mean_magnitude','edge_mean','bright_mean']].describe())


       flow_mean_magnitude   edge_mean  bright_mean
count           200.000000  200.000000   200.000000
mean              4.505264    0.039800    86.767974
std               2.076349    0.026282    38.138897
min               0.101223    0.001816    10.620280
25%               3.079040    0.018785    47.695731
50%               4.486717    0.034969    97.596355
75%               5.856686    0.057665   113.264551
max              13.410395    0.120019   166.878093
